# Job: Ingestão para a Landing

1. Baixa os dados brutos da MUNIC da API do IBGE
2. Salva no schema landing no formato .xlsx (a forma original)

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions


In [0]:
import requests
import os

In [0]:
# print(f"Origem : {munic_source_url}")
# print(f"Destino: {munic_file}")


In [0]:
def download_file(munic_file, munic_source_url):
    # Verifica se o arquivo já existe
    if os.path.exists(munic_file):
        print(f"Arquivo já existe em {munic_file} — download ignorado.")
        return False
    else:
        # Faz a requisição à API do IBGE
        file = requests.get(munic_source_url)

        # Lança uma exceção caso ocorra um erro na requisição
        file.raise_for_status()

        # Grava o conteúdo da resposta no arquivo xlsx
        with open(munic_file, "wb") as f:
            f.write(file.content)
        
        print(f"Arquivo baixado com sucesso em {munic_file}")
        return True


In [0]:
# Para cada edição registrada em munic_source_url_map, verifica se o arquivo já existe na landing
# Caso não exista, baixa do IBGE e salva no caminho definido em munic_file_paths.
# Ao final, a edição é verificada (verificar_arquivo_landing, env_functions: SHA-256
# recalculado sempre, aviso de divergência contra o último declarado, baixado_em
# honesto) e o notebook declara a linha 'landing' no gravar_linha_manifest — o
# mesmo padrão de declaração de todas as etapas do pipeline.

for year, url in munic_source_url_map.items():
    # Separa visualmente o output de cada arquivo
    print(f"\n{'-' * 70}\nEdição {year}\n{'-' * 70}")

    file_path = munic_file_paths[year]
    baixou = download_file(file_path, url)   # False se o XLSX já existia (idempotente)

    # Verifica o arquivo da edição e monta os valores da linha 'landing' no pipeline_manifest
    file_sha256, detalhes_edicao = verificar_arquivo_landing(year, url, file_path, baixou)

    gravar_linha_manifest(
        etapa="landing",
        ano_edicao=year,
        tabela_destino=landing_folder_path,   # caminho do volume (não é tabela)
        sha256_xlsx=file_sha256,
        detalhes=detalhes_edicao,
    )
